# ARC + MolFormer — ChEMBL & BindingDB Pipeline (v9)
## Adaptive Retention & Correction for Molecular Target Classification

### Based on:
- **Paper**: "Adaptive Retention & Correction: Test-Time Training for Continual Learning" (ICLR 2025)
- **Backbone**: IBM MolFormer-XL (`ibm/MoLFormer-XL-both-10pct`)
- **Datasets**: ChEMBL 15-class (`chembl_15class_final.csv`) + BindingDB 15-class (`bindingdb_15class.csv`)

---

### ARC Core Idea (Paper Summary)
| Component | What it does |
|-----------|-------------|
| **OTD** | Out-of-Task Detection — identifies whether each test sample is from a past task or current task using confidence + predicted class |
| **Adaptive Retention** | If OTD detects a correctly-classified past-task sample (high confidence, past class), do ONE gradient update on the classifier head: L = L_CE + L_EM |
| **Adaptive Correction** | If OTD detects a misclassified past-task sample (predicted into current task with low relative confidence), use Task-based Softmax Score (TSS) to reassign prediction |

### Pipeline Architecture
```
ChEMBL / BindingDB CSV
    → SMILES Cleaning (RDKit)
    → Scaffold Split (Bemis-Murcko, stratified per-class)
    → MolFormer Feature Extraction (mean-pooled embeddings, frozen backbone)
    → CIL Task Construction (15 classes → 5 tasks × 3 classes/task)
    → Train Expanding Linear Head (per task)
    → Evaluate: Baseline vs ARC
    → Metrics: Avg Accuracy (AB), Forgetting (F), AUROC, BWT, FWT
```

### Key Design Decisions (v9, adapted from v8):
| # | What | Why |
|---|------|-----|
| v9-1 | `classes_per_task=3` → 5 tasks on 15 classes | Matches 15-class datasets; paper used 2/task on 14 ATC |
| v9-2 | Dominant-label assignment kept | Both datasets are already single-label; no multi-label parsing needed |
| v9-3 | Stratified scaffold split within each class | Prevents class leakage; same fix as v8 FIX-1 |
| v9-4 | MolFormer backbone frozen throughout | ARC paper Sec 3.2: representation layer retains knowledge; only classifier drifts |
| v9-5 | ARC runs memory-free; Baseline uses replay | ARC paper design: replay suppresses the very classifier bias ARC corrects |
| v9-6 | Persistent clf_arc across eval calls | Retention updates accumulate; not reset on each task evaluation |
| v9-7 | Batch theta computed once per test batch | Not per-sample (which trivially equals itself) |
| v9-8 | TSS denominator = prefix logits only | BUG-B fix from v7: using all classes suppressed older-task scores |


## Step 1 — Install & Imports

In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'torch>=2.0.0',
    'transformers>=4.35.0',
    'scikit-learn>=1.3.0',
    'numpy>=1.24.0',
    'matplotlib>=3.7.0',
    'seaborn>=0.12.0',
    'pandas>=2.0.0',
    'rdkit',
    'einops',
    'rotary-embedding-torch',
    'scipy',
    'tqdm',
]
for pkg in packages:
    try:
        install(pkg)
        print(f'  ✓ {pkg}')
    except Exception as e:
        print(f'  ✗ {pkg}: {e}')
print('\nInstallation complete.')


  ✓ torch>=2.0.0


  ✓ transformers>=4.35.0


  ✓ scikit-learn>=1.3.0


  ✓ numpy>=1.24.0


  ✓ matplotlib>=3.7.0


...

  ✓ rdkit


  ✓ einops


  ✓ rotary-embedding-torch


  ✓ scipy


  ✓ tqdm

Installation complete.


In [2]:
import os, json, copy, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import MolStandardize, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModel

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print('All imports OK')


Device: cuda
All imports OK


## Step 2 — Configuration

In [43]:
CFG = {
    # ── Data paths ────────────────────────────────────────────────────────
    'chembl_path'     : 'chembl_15class_final.csv',    # col: smiles, label (0-14)
    'bindingdb_path'  : 'bindingdb_15class.csv',       # col: smiles, label (0-14)
    'output_dir'      : 'arc_output_v9',

    # ── SMILES cleaning ───────────────────────────────────────────────────
    'min_ha'          : 5,
    'max_ha'          : 100,
    'min_mw'          : 100,
    'max_mw'          : 1500,

    # ── Sampling (set None to use full dataset, or e.g. 20000 for quick runs) ─
    # With ~1M rows each, full extraction is slow. For prototyping set to 30000.
    # For publication-quality results set to None.
    'max_samples'     : 30000,   # ← change to None for full run
    'seed'            : 42,

    # ── Train/val/test split ──────────────────────────────────────────────
    'train_ratio'     : 0.70,
    'val_ratio'       : 0.15,
    'test_ratio'      : 0.15,

    # ── CIL tasks ─────────────────────────────────────────────────────────
    # 15 classes / 3 per task = 5 tasks (paper-compatible design)
    'classes_per_task': 3,

    # ── MolFormer ─────────────────────────────────────────────────────────
    'molformer_name'  : 'ibm/MoLFormer-XL-both-10pct',
    'molformer_batch' : 32,
    'max_smiles_len'  : 202,

    # ── Classifier training ───────────────────────────────────────────────
    'clf_epochs'      : 100,
    'clf_lr'          : 1e-3,
    'clf_batch'       : 64,
    'es_patience'     : 10,
    'es_min_delta'    : 1e-4,

    # ── ARC hyperparameters ───────────────────────────────────────────────
    # epsilon: Assumption 1 — past-class prediction accepted as correct only
    #   when confidence >= epsilon. 0.50 is well-calibrated for MolFormer.
    'arc_epsilon'     : 0.20,
    # theta: Assumption 2 — w = c/c_hat < theta → correction fires.
    #   0.40 = current-task confidence must be < 40% of past-task confidence.
    'arc_theta'       : 1.5,
    # arc_temp: TSS temperature T > 1 (T^(t-i) in paper Definition 1)
    'arc_temp'        : 2.0,
    'arc_lr'          : 5e-4,
}

os.makedirs(CFG['output_dir'], exist_ok=True)
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

print('Config ready')
print(f"  classes_per_task = {CFG['classes_per_task']}  →  {15 // CFG['classes_per_task']} tasks on 15 classes")
print(f"  arc_epsilon={CFG['arc_epsilon']}  arc_theta={CFG['arc_theta']}  arc_temp={CFG['arc_temp']}")
print(f"  max_samples={CFG['max_samples']}  (set None for full ~1M-row run)")


Config ready
  classes_per_task = 3  →  5 tasks on 15 classes
  arc_epsilon=0.2  arc_theta=1.5  arc_temp=2.0
  max_samples=30000  (set None for full ~1M-row run)


## Step 3 — Data Loading & SMILES Cleaning

Both ChEMBL and BindingDB CSVs have identical schema: `smiles` (str) + `label` (int 0–14).
We apply the same RDKit standardisation pipeline used in the original v8 notebook:
1. Largest fragment selection (de-salt)
2. Uncharging
3. Tautomer canonicalisation
4. Heavy-atom / MW filter
5. Canonical SMILES generation


In [6]:
def clean_smiles(smiles, cfg):
    """Full SMILES cleaning: missing → invalid → salt → standardize → filter → canon."""
    if not isinstance(smiles, str) or smiles.strip() == '':
        return None, 'missing_or_empty'
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None:
        return None, 'invalid_smiles'
    try:
        mol = MolStandardize.rdMolStandardize.LargestFragmentChooser().choose(mol)
    except Exception:
        pass
    try:
        mol = MolStandardize.rdMolStandardize.Uncharger().uncharge(mol)
        mol = MolStandardize.rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
    except Exception:
        pass
    try:
        ha = mol.GetNumHeavyAtoms()
        mw = rdMolDescriptors.CalcExactMolWt(mol)
        if not (cfg['min_ha'] <= ha <= cfg['max_ha'] and cfg['min_mw'] <= mw <= cfg['max_mw']):
            return None, 'filtered_out'
    except Exception:
        return None, 'filter_error'
    canon = Chem.MolToSmiles(mol, canonical=True)
    return (canon, 'ok') if canon else (None, 'canon_failed')


def get_scaffold(smiles):
    """Bemis-Murcko scaffold; fallback = original smiles."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return smiles
        sc = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(sc, canonical=True)
    except Exception:
        return smiles


print('SMILES cleaning helpers defined')


SMILES cleaning helpers defined


In [7]:
def load_and_clean_dataset(path, cfg, dataset_name='Dataset'):
    """
    Load a CSV with columns [smiles, label], optionally subsample,
    clean SMILES, compute scaffolds, and return a cleaned DataFrame.

    Subsampling strategy: stratified random sample (equal per class)
    so class balance is preserved exactly.
    """
    print(f'\n=== Loading {dataset_name} from {path} ===')
    df = pd.read_csv(path)
    print(f'  Raw shape: {df.shape}')
    print(f'  Labels: {sorted(df["label"].unique())}')

    # ── Stratified subsample ─────────────────────────────────────────────
    if cfg.get('max_samples') is not None:
        n_classes = df['label'].nunique()
        per_class = cfg['max_samples'] // n_classes
        rng = np.random.default_rng(cfg['seed'])
        frames = []
        for cls in sorted(df['label'].unique()):
            cls_df = df[df['label'] == cls]
            n_take = min(per_class, len(cls_df))
            idx    = rng.choice(len(cls_df), size=n_take, replace=False)
            frames.append(cls_df.iloc[idx])
        df = pd.concat(frames).reset_index(drop=True)
        print(f'  After stratified subsample ({cfg["max_samples"]} total): {df.shape}')

    # ── Clean SMILES ─────────────────────────────────────────────────────
    res = df['smiles'].apply(lambda s: clean_smiles(s, cfg))
    df['canon_smiles'] = res.apply(lambda x: x[0])
    df['clean_status'] = res.apply(lambda x: x[1])
    print(f'\n  Cleaning report ({dataset_name}):')
    print(df['clean_status'].value_counts().to_string())

    df = df[df['clean_status'] == 'ok'].drop_duplicates('canon_smiles').copy()
    df = df.rename(columns={'label': 'label_id'})
    print(f'  After cleaning: {df.shape}')

    # ── Scaffold ─────────────────────────────────────────────────────────
    df['scaffold'] = df['canon_smiles'].apply(get_scaffold)

    print(f'  Class distribution:')
    print(df['label_id'].value_counts().sort_index().to_string())
    return df


chembl_df   = load_and_clean_dataset(CFG['chembl_path'],   CFG, 'ChEMBL-15')
bindingdb_df = load_and_clean_dataset(CFG['bindingdb_path'], CFG, 'BindingDB-15')



=== Loading ChEMBL-15 from chembl_15class_final.csv ===
  Raw shape: (1103220, 2)
  Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  After stratified subsample (30000 total): (30000, 2)

  Cleaning report (ChEMBL-15):
clean_status
ok              29699
filtered_out      301
  After cleaning: (29681, 4)
  Class distribution:
label_id
0     1960
1     1981
2     1979
3     1971
4     1983
5     1980
6     1988
7     1975
8     1980
9     1975
10    1982
11    1976
12    1988
13    1986
14    1977

=== Loading BindingDB-15 from bindingdb_15class.csv ===
  Raw shape: (914051, 2)
  Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
  After stratified subsample (30000 total): (30000, 2)

  Cleaning report (BindingDB-15):
clean_status
ok                29492
invalid_smiles      313
filtered_out        195
  After cleaning: (29466, 4)
  Class distribution:
label_id
0     1947
1     1964
2     1964
3     1964
4     1963
5     1969
6     1962
7     1961
8     1960
9     1968

## Step 4 — Label Maps & Stratified Scaffold Split

Since both datasets are already single-label (label 0–14), no multi-label parsing is needed.
We build a label map and perform **per-class stratified scaffold split** (Bemis-Murcko).

**Why scaffold split?** Random split leaks similar molecules into train and test,
inflating test accuracy. Scaffold split ensures structural novelty in the test set,
which better reflects real-world generalisation.


In [8]:
def stratified_scaffold_split(df, label_col, train_r, val_r, test_r, seed=42):
    """
    Stratified scaffold split: split within each class independently.
    Guarantees every class appears in all three splits. 
    """
    assert abs(train_r + val_r + test_r - 1.0) < 1e-6
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []

    for cls in sorted(df[label_col].unique()):
        cls_df = df[df[label_col] == cls]
        sc2idx = defaultdict(list)
        for idx, sc in zip(cls_df.index, cls_df['scaffold']):
            sc2idx[sc].append(idx)
        groups = list(sc2idx.values())
        rng.shuffle(groups)
        n, n_tr, n_vl = len(cls_df), max(1, int(len(cls_df) * train_r)), max(1, int(len(cls_df) * val_r))
        cls_tr, cls_vl, cls_ts = [], [], []
        for g in groups:
            if   len(cls_tr) < n_tr: cls_tr.extend(g)
            elif len(cls_vl) < n_vl: cls_vl.extend(g)
            else:                    cls_ts.extend(g)
        # Safety: ensure val and test each have >= 1 sample
        if len(cls_vl) == 0 and len(cls_tr) > 1: cls_vl.append(cls_tr.pop())
        if len(cls_ts) == 0 and len(cls_tr) > 1: cls_ts.append(cls_tr.pop())
        train_idx.extend(cls_tr); val_idx.extend(cls_vl); test_idx.extend(cls_ts)

    tr, vl, ts = df.loc[train_idx].copy(), df.loc[val_idx].copy(), df.loc[test_idx].copy()
    for sname, sdf in [('val', vl), ('test', ts)]:
        missing = set(df[label_col].unique()) - set(sdf[label_col].unique())
        if missing:
            print(f'  WARNING: {sname} missing classes {missing}')
    return tr, vl, ts


def verify_split(train_df, val_df, test_df, label_col='label_id', name=''):
    print(f'  {name} splits:')
    for sname, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
        counts = sdf[label_col].value_counts().sort_index()
        print(f'    {sname}: {len(sdf)} samples | min_cls={counts.min()} max_cls={counts.max()}')
    tsc = set(train_df['scaffold']); vsc = set(val_df['scaffold']); esc = set(test_df['scaffold'])
    print(f'    Scaffold leakage → Train∩Val={len(tsc&vsc)} Train∩Test={len(tsc&esc)} Val∩Test={len(vsc&esc)}')


def build_label_map(df, label_col='label_id'):
    """Build {encoded_id: 'Target-X'} label map."""
    return {int(c): f'Target-{c}' for c in sorted(df[label_col].unique())}


# ── ChEMBL ────────────────────────────────────────────────────────────────────
print('\n=== ChEMBL Scaffold Split ===')
chembl_train, chembl_val, chembl_test = stratified_scaffold_split(
    chembl_df, 'label_id',
    CFG['train_ratio'], CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
)
verify_split(chembl_train, chembl_val, chembl_test, name='ChEMBL')
chembl_label_map = build_label_map(chembl_df)
print(f'  Label map: {chembl_label_map}')

# ── BindingDB ─────────────────────────────────────────────────────────────────
print('\n=== BindingDB Scaffold Split ===')
bindingdb_train, bindingdb_val, bindingdb_test = stratified_scaffold_split(
    bindingdb_df, 'label_id',
    CFG['train_ratio'], CFG['val_ratio'], CFG['test_ratio'], CFG['seed']
)
verify_split(bindingdb_train, bindingdb_val, bindingdb_test, name='BindingDB')
bindingdb_label_map = build_label_map(bindingdb_df)
print(f'  Label map: {bindingdb_label_map}')



=== ChEMBL Scaffold Split ===
  ChEMBL splits:
    Train: 20777 samples | min_cls=1372 max_cls=1394
    Val: 4449 samples | min_cls=294 max_cls=299
    Test: 4455 samples | min_cls=294 max_cls=299
    Scaffold leakage → Train∩Val=646 Train∩Test=616 Val∩Test=193
  Label map: {0: 'Target-0', 1: 'Target-1', 2: 'Target-2', 3: 'Target-3', 4: 'Target-4', 5: 'Target-5', 6: 'Target-6', 7: 'Target-7', 8: 'Target-8', 9: 'Target-9', 10: 'Target-10', 11: 'Target-11', 12: 'Target-12', 13: 'Target-13', 14: 'Target-14'}

=== BindingDB Scaffold Split ===
  BindingDB splits:
    Train: 20622 samples | min_cls=1363 max_cls=1381
    Val: 4421 samples | min_cls=292 max_cls=299
    Test: 4423 samples | min_cls=292 max_cls=297
    Scaffold leakage → Train∩Val=712 Train∩Test=709 Val∩Test=215
  Label map: {0: 'Target-0', 1: 'Target-1', 2: 'Target-2', 3: 'Target-3', 4: 'Target-4', 5: 'Target-5', 6: 'Target-6', 7: 'Target-7', 8: 'Target-8', 9: 'Target-9', 10: 'Target-10', 11: 'Target-11', 12: 'Target-12', 13: 

## Step 5 — CIL Task Construction

We build **cumulative Class-Incremental Learning (CIL)** tasks:
- 15 classes / 3 per task → **5 tasks**
- Task *t* trains on all classes seen up to *t* (cumulative)
- Evaluation at task *t* covers all tasks 0..t (to measure forgetting)

This matches the **class-incremental** setting from the ARC paper.


In [9]:
def make_cil_tasks(train_df, val_df, test_df, cpt, label_map):
    """Build cumulative CIL tasks. cpt = classes per task."""
    all_ids = sorted(train_df['label_id'].unique())
    tasks = []
    for t in range(int(np.ceil(len(all_ids) / cpt))):
        new_cls  = all_ids[t*cpt : (t+1)*cpt]
        seen_cls = all_ids[: (t+1)*cpt]
        tasks.append({
            'task_id'    : t,
            'new_classes': new_cls,
            'all_classes': list(seen_cls),
            'n_classes'  : len(seen_cls),
            'class_names': {lid: label_map[lid] for lid in seen_cls},
            'train': train_df[train_df['label_id'].isin(seen_cls)].copy(),
            'val'  : val_df[val_df['label_id'].isin(seen_cls)].copy(),
            'test' : test_df[test_df['label_id'].isin(seen_cls)].copy(),
        })
    return tasks


chembl_tasks   = make_cil_tasks(chembl_train,   chembl_val,   chembl_test,   CFG['classes_per_task'], chembl_label_map)
bindingdb_tasks = make_cil_tasks(bindingdb_train, bindingdb_val, bindingdb_test, CFG['classes_per_task'], bindingdb_label_map)

print(f'ChEMBL CIL tasks: {len(chembl_tasks)}')
for t in chembl_tasks:
    print(f"  Task {t['task_id']}: new={[t['class_names'][c] for c in t['new_classes']]}  "          f"train={len(t['train'])} val={len(t['val'])} test={len(t['test'])}")

print(f'\nBindingDB CIL tasks: {len(bindingdb_tasks)}')
for t in bindingdb_tasks:
    print(f"  Task {t['task_id']}: new={[t['class_names'][c] for c in t['new_classes']]}  "          f"train={len(t['train'])} val={len(t['val'])} test={len(t['test'])}")


ChEMBL CIL tasks: 5
  Task 0: new=['Target-0', 'Target-1', 'Target-2']  train=4143 val=887 test=890
  Task 1: new=['Target-3', 'Target-4', 'Target-5']  train=8296 val=1776 test=1782
  Task 2: new=['Target-6', 'Target-7', 'Target-8']  train=12455 val=2669 test=2673
  Task 3: new=['Target-9', 'Target-10', 'Target-11']  train=16609 val=3558 test=3563
  Task 4: new=['Target-12', 'Target-13', 'Target-14']  train=20777 val=4449 test=4455

BindingDB CIL tasks: 5
  Task 0: new=['Target-0', 'Target-1', 'Target-2']  train=4112 val=880 test=883
  Task 1: new=['Target-3', 'Target-4', 'Target-5']  train=8239 val=1763 test=1769
  Task 2: new=['Target-6', 'Target-7', 'Target-8']  train=12357 val=2645 test=2652
  Task 3: new=['Target-9', 'Target-10', 'Target-11']  train=16491 val=3533 test=3539
  Task 4: new=['Target-12', 'Target-13', 'Target-14']  train=20622 val=4421 test=4423


## Step 6 — MolFormer Feature Extraction

**Key insight from ARC paper (Section 3.2):** In pretrained models, the representation
layer retains knowledge extremely well — catastrophic forgetting comes almost entirely
from the **classifier head bias**. Therefore we:
1. Freeze MolFormer backbone completely
2. Extract mean-pooled embeddings once (not per-task)
3. Only the linear classifier head is trained/adapted

**Mean pooling** (not CLS token): MolFormer-XL is trained with mean pooling over
non-padding tokens. Using CLS gives suboptimal molecular representations.


In [10]:
print('Loading MolFormer tokenizer & model...')
tokenizer = AutoTokenizer.from_pretrained(CFG['molformer_name'], trust_remote_code=True)
molformer = AutoModel.from_pretrained(
    CFG['molformer_name'], trust_remote_code=True, deterministic_eval=True
)
molformer.eval().to(DEVICE)
FEAT_DIM = molformer.config.hidden_size

# Freeze ALL backbone parameters — ARC paper Sec 3.2 assumption
for p in molformer.parameters():
    p.requires_grad = False

n_frozen = sum(1 for p in molformer.parameters() if not p.requires_grad)
n_total  = sum(1 for p in molformer.parameters())
print(f'MolFormer loaded on {DEVICE}  |  hidden_dim={FEAT_DIM}')
print(f'Backbone frozen: {n_frozen}/{n_total} params (ARC paper: representation layer does NOT forget)')


Loading MolFormer tokenizer & model...
MolFormer loaded on cuda  |  hidden_dim=768
Backbone frozen: 195/195 params (ARC paper: representation layer does NOT forget)


In [11]:
@torch.no_grad()
def extract_molformer_features(smiles_list, batch_size=32):
    """
    Extract mean-pooled MolFormer embeddings over non-padding tokens.
    Returns np.ndarray of shape (N, hidden_dim).

    Mean pooling (not CLS): matches MolFormer-XL training objective.
    Reference: Ross et al. 2022 (MolFormer paper) + ibm/MoLFormer-XL-both-10pct model card.
    """
    all_emb = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc='MolFormer encode'):
        batch = smiles_list[i: i + batch_size]
        enc   = tokenizer(
            batch, padding=True, truncation=True,
            max_length=CFG['max_smiles_len'], return_tensors='pt'
        ).to(DEVICE)
        out    = molformer(**enc)
        hidden = out.last_hidden_state          # (B, L, D)
        mask   = enc['attention_mask']          # (B, L)
        mask_f = mask.unsqueeze(-1).float()     # (B, L, 1)
        summed = (hidden * mask_f).sum(dim=1)   # (B, D)
        counts = mask_f.sum(dim=1).clamp(min=1) # (B, 1)
        pooled = (summed / counts).cpu().numpy()
        all_emb.append(pooled)
    return np.vstack(all_emb)


print('Feature extractor ready')


Feature extractor ready


In [12]:
def extract_dataset_features(df, train_df, val_df, test_df, dataset_name='Dataset'):
    """Extract features for all molecules once, then index by split."""
    print(f'\n=== Extracting {dataset_name} features ({len(df)} molecules) ===')
    feats_all = extract_molformer_features(df['canon_smiles'].tolist(), CFG['molformer_batch'])
    idx_to_feat = {idx: feat for idx, feat in zip(df.index, feats_all)}

    def get_feats(split_df):
        X = np.stack([idx_to_feat[i] for i in split_df.index])
        y = split_df['label_id'].values
        return X, y

    X_tr, y_tr = get_feats(train_df)
    X_vl, y_vl = get_feats(val_df)
    X_te, y_te = get_feats(test_df)
    print(f'  train:{X_tr.shape} | val:{X_vl.shape} | test:{X_te.shape}')
    return X_tr, y_tr, X_vl, y_vl, X_te, y_te


chembl_X_tr, chembl_y_tr, chembl_X_vl, chembl_y_vl, chembl_X_te, chembl_y_te = \
    extract_dataset_features(chembl_df, chembl_train, chembl_val, chembl_test, 'ChEMBL')

bindingdb_X_tr, bindingdb_y_tr, bindingdb_X_vl, bindingdb_y_vl, bindingdb_X_te, bindingdb_y_te = \
    extract_dataset_features(bindingdb_df, bindingdb_train, bindingdb_val, bindingdb_test, 'BindingDB')



=== Extracting ChEMBL features (29681 molecules) ===


MolFormer encode:   0%|          | 0/928 [00:00<?, ?it/s]

  train:(20777, 768) | val:(4449, 768) | test:(4455, 768)

=== Extracting BindingDB features (29466 molecules) ===


MolFormer encode:   0%|          | 0/921 [00:00<?, ?it/s]

  train:(20622, 768) | val:(4421, 768) | test:(4423, 768)


## Step 7 — Expanding Linear Classifier & Replay Buffer

The ARC paper uses a simple linear classifier h_cls(f(x)) where:
- `f` = frozen backbone (MolFormer)
- `h_cls` = linear layer that **expands** as new classes are added each task

**Expanding head**: when task t introduces new classes, new output neurons are appended
with Xavier initialisation, preserving old class weights exactly.

**Replay buffer** (baseline only): stores exemplars via reservoir sampling for
experience replay. ARC runs WITHOUT replay by design (ARC paper, memory-free setting).


In [13]:
class MolDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


class ExpandingLinearHead(nn.Module):
    """
    Single linear layer that grows to accommodate new classes each task.
    Matches ARC paper h_ωcls.
    """
    def __init__(self, in_dim, n_init):
        super().__init__()
        self.fc = nn.Linear(in_dim, n_init)

    def grow(self, n_new, device):
        """Append n_new output neurons, preserving old class weights."""
        old   = self.fc
        n_old = old.out_features
        new_fc = nn.Linear(old.in_features, n_old + n_new)
        with torch.no_grad():
            new_fc.weight[:n_old] = old.weight
            new_fc.bias[:n_old]   = old.bias
            nn.init.xavier_uniform_(new_fc.weight[n_old:])
            nn.init.zeros_(new_fc.bias[n_old:])
        self.fc = new_fc.to(device)

    def forward(self, x): return self.fc(x)

    @property
    def n_classes(self): return self.fc.out_features


class ReplayBuffer:
    """
    Reservoir-sampling exemplar memory.
    Stores up to max_per_class feature vectors per class.
    Used ONLY by the baseline (not ARC).
    """
    def __init__(self, max_per_class):
        self.max_per_class = max_per_class
        self._X = {}; self._y = {}; self._counts = {}

    def update(self, X, y, rng=None):
        if rng is None: rng = np.random.default_rng()
        for cls in np.unique(y):
            mask = y == cls
            if cls not in self._X:
                self._X[cls] = []; self._y[cls] = []; self._counts[cls] = 0
            for feat in X[mask]:
                n = self._counts[cls]
                if n < self.max_per_class:
                    self._X[cls].append(feat); self._y[cls].append(int(cls))
                else:
                    j = int(rng.integers(0, n + 1))
                    if j < self.max_per_class:
                        self._X[cls][j] = feat
                self._counts[cls] += 1

    def sample(self):
        if not self._X: return None, None
        X_p = np.vstack([np.stack(self._X[c]) for c in sorted(self._X)])
        y_p = np.concatenate([np.array(self._y[c]) for c in sorted(self._y)])
        return X_p, y_p

    def __len__(self): return sum(len(v) for v in self._X.values())


print('ExpandingLinearHead and ReplayBuffer defined')


ExpandingLinearHead and ReplayBuffer defined


In [42]:
def train_classifier_on_task(clf, X_tr, y_tr, X_val, y_val,
                              seen_cls, epochs, lr, batch_size, device,
                              es_patience=10, es_min_delta=1e-4):
    tr_mask  = np.isin(y_tr, seen_cls)
    X_t, y_t = X_tr[tr_mask], y_tr[tr_mask]
    val_mask = np.isin(y_val, seen_cls)
    X_v, y_v = X_val[val_mask], y_val[val_mask]

    n_total_cls  = clf.n_classes
    class_counts = np.array([max(1, (y_t == c).sum()) for c in range(n_total_cls)])
    class_weights = 1.0 / class_counts.astype(float)
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

    ds      = MolDataset(X_t, y_t)
    dl      = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)
    opt     = torch.optim.Adam(clf.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=weight_tensor)

    best_val_acc, best_weights, patience_count = -1.0, copy.deepcopy(clf.state_dict()), 0

    Xv_t = torch.tensor(X_v, dtype=torch.float32).to(device)
    yv_t = torch.tensor(y_v, dtype=torch.long).to(device)

    for epoch in range(epochs):
        clf.train()
        for Xb, yb in dl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss_fn(clf(Xb), yb).backward()
            opt.step()

        clf.eval()
        with torch.no_grad():
            val_acc = (clf(Xv_t).argmax(1) == yv_t).float().mean().item()

        if val_acc > best_val_acc + es_min_delta:
            best_val_acc = val_acc; best_weights = copy.deepcopy(clf.state_dict()); patience_count = 0
        else:
            patience_count += 1
        if patience_count >= es_patience:
            break

    clf.load_state_dict(best_weights)
    clf.eval()
    return clf

print('train_classifier_on_task defined')

train_classifier_on_task defined


## Step 8 — ARC: Out-of-Task Detection (OTD)

**Algorithm 1 from the paper:**

```
if predicted class ∈ PAST tasks AND confidence ≥ ε:
    → RETENTION  (Assumption 1: correctly classified past sample)

elif predicted class ∈ CURRENT task AND w = c/ĉ < θ:
    → CORRECTION  (Assumption 2: misclassified past sample)

else:
    → accept as current-task prediction
```

Where:
- `c` = max softmax confidence of the predicted class
- `ĉ` = max confidence over PAST-task class logits only  
- `w = c / ĉ` — ratio < θ means the model is more confident it's from a past task

**Batch theta** (FIX-2 from v8): θ is adapted from the full test batch median(w) × 0.8,
computed ONCE before the per-sample loop. Original per-sample computation was trivially
self-referential (median of 1 value = itself).


In [15]:
def compute_batch_theta(logits_all, task_id, n_classes_per_task, theta):
    """
    Adaptive theta from the full test batch — computed ONCE before per-sample loop.
    Returns min(theta, 0.8 * median_w) over all test samples.

    FIX-2 (v8): previous code computed this inside the per-sample loop on B=1,
    making median(w) == w[0] always — no real adaptation.
    """
    s = n_classes_per_task
    past_boundary = s * task_id
    if past_boundary == 0:
        return theta
    with torch.no_grad():
        probs     = F.softmax(logits_all.float(), dim=-1)
        conf_all  = probs.max(dim=-1).values
        c_hat_all = probs[:, :past_boundary].max(dim=-1).values
        w_all     = conf_all / (c_hat_all + 1e-8)
    adaptive = float(w_all.median()) * 0.8
    return min(theta, adaptive)


def out_of_task_detection(logits, task_id, n_classes_per_task, epsilon, theta, batch_theta=None):
    """
    OTD: classify each sample as 'retention' / 'correction' / 'current'.
    Implements Algorithm 1 from the ARC paper.

    FIX-1 (v8): OTD logic was INVERTED in earlier versions.
    CORRECT logic:
      Assumption 1: pred ∈ PAST classes AND conf >= ε  → RETENTION
      Assumption 2: pred ∈ CURRENT classes AND w < θ  → CORRECTION
      Otherwise: 'current'

    Parameters
    ----------
    logits       : Tensor [B, n_total_classes]
    task_id      : int  (0-based current task)
    n_classes_per_task : int  s
    epsilon      : float  Assumption 1 threshold
    theta        : float  Assumption 2 base threshold
    batch_theta  : float|None  pre-computed adaptive theta (overrides theta if given)

    Returns
    -------
    decisions : list['retention'|'correction'|'current']
    probs     : Tensor [B, C]
    pred_cls  : Tensor [B]
    """
    s = n_classes_per_task
    past_boundary = s * task_id  # class indices 0..past_boundary-1 = past tasks

    probs    = F.softmax(logits.float(), dim=-1)
    pred_cls = probs.argmax(dim=-1)
    conf_all = probs.max(dim=-1).values

    effective_theta = batch_theta if batch_theta is not None else theta

    decisions = []
    for i in range(logits.shape[0]):
        pred_i = pred_cls[i].item()
        c      = conf_all[i].item()

        if past_boundary == 0:
            decisions.append('current')
            continue

        if pred_i < past_boundary:
            # Predicted into a PAST-task class
            if c >= epsilon:
                decisions.append('retention')   # Assumption 1: high-conf past → retain
            else:
                decisions.append('current')     # Low-conf past → too uncertain
        else:
            # Predicted into CURRENT-task class
            c_hat = probs[i, :past_boundary].max().item()
            w     = c / (c_hat + 1e-8)
            if w < effective_theta:
                decisions.append('correction')  # Assumption 2: current conf << past conf
            else:
                decisions.append('current')

    return decisions, probs, pred_cls


print('OTD (Out-of-Task Detection) defined')


OTD (Out-of-Task Detection) defined


## Step 9 — Adaptive Retention

**Equation 2 & 3 from the paper:**
```
L_CE = -ŷ_c log p_c                    (pseudo-label cross-entropy)
L_EM = -Σ p_i log p_i                  (entropy minimisation)
L    = L_CE + L_EM                     (combined objective)
```

**One gradient update per sample** — matches the online inference setting of the paper.
Entropy minimisation reduces noise from pseudo-label uncertainty.


In [16]:
def adaptive_retention_step(clf, x_single, pseudo_label, lr, device):
    """
    One gradient update on the classifier head using pseudo-label supervision.
    L = L_CE(pseudo_label) + L_EM  (Eq. 2 & 3, ARC paper).

    Called from grad-ENABLED scope.
    x_single.detach().clone() creates a fresh leaf tensor to allow
    loss.backward() to compute gradients through clf.fc.
    """
    clf.train()
    opt = torch.optim.SGD(clf.parameters(), lr=lr)
    opt.zero_grad()

    x = x_single.detach().clone().to(device)
    logits  = clf(x)                              # (1, C)
    probs   = F.softmax(logits, dim=-1)
    label_t = torch.tensor([pseudo_label], dtype=torch.long).to(device)

    L_CE = F.cross_entropy(logits, label_t)
    # clamp inside log only — shifting prob mass distorts entropy
    L_EM = -(probs * probs.clamp(min=1e-8).log()).sum(dim=-1).mean()

    loss = L_CE + L_EM
    loss.backward()
    opt.step()
    clf.eval()


print('Adaptive Retention defined')


Adaptive Retention defined


## Step 10 — Adaptive Correction (TSS)

**Definition 1 from the paper — Task-based Softmax Score (TSS):**

$$S_i = \frac{\max_{s(i-1) \le k < si} \exp(z_k / T^{t-i})}{\sum_{j=0}^{si-1} \exp(z_j / T^{t-i})}$$

Key design choices:
- **Per-task temperature scaling**: T^(t-i) — current task gets T^0=1, older tasks larger exponent
- **Prefix denominator** (BUG-B fix from v7): denominator sums over logits[0..s*i], NOT all classes.
  Using all classes suppresses older-task scores relative to current task → correction becomes no-op.
- Prediction is reassigned to the class with highest TSS score across all past tasks


In [17]:
def compute_tss(logits_single, task_id, n_classes_per_task, temperature):
    """
    Task-based Softmax Score (TSS) — Definition 1 from ARC paper.

    BUG-B fix (v7→v8): denominator uses prefix logits [0..s*i), NOT full logits.
      Full denominator suppresses every older-task S_i → correction always picks
      current task → ARC identical to baseline.

    BUG-C fix (v7→v8): per-task temperature exponent T^(t-i).
      Current task (i == t) gets T^0 = 1.
      Older tasks get larger exponent → less spread → not unfairly suppressed.

    Parameters
    ----------
    logits_single      : Tensor [n_total_classes]  (1-D, one sample)
    task_id            : int  0-based current task (t = task_id + 1, 1-based)
    n_classes_per_task : int  s
    temperature        : float  T > 1 recommended

    Returns
    -------
    (best_task_idx, best_class_idx) : ints (0-based)
    """
    s = n_classes_per_task
    t = task_id + 1  # 1-based current task
    z = logits_single.float()

    best_score, best_task, best_cls = -float('inf'), 0, 0

    for i in range(1, t + 1):            # i = 1 .. t (1-based)
        exponent  = t - i                # T^(t-i): current=0, oldest=t-1
        temp_i    = temperature ** exponent

        # Numerator: max exp(z_k/T^(t-i)) for k in [s*(i-1), s*i)
        task_logits = z[s * (i - 1) : s * i]
        scaled_task = task_logits / temp_i
        numerator   = scaled_task.exp().max()

        # Denominator: sum exp(z_j/T^(t-i)) for j in [0, s*i)  ← BUG-B fix
        prefix_logits = z[: s * i]
        denominator   = (prefix_logits / temp_i).exp().sum()

        score = (numerator / (denominator + 1e-12)).item()

        if score > best_score:
            best_score = score
            best_task  = i - 1                                     # back to 0-based
            best_cls   = s * (i - 1) + task_logits.argmax().item()

    return best_task, best_cls


print('Adaptive Correction (TSS) defined')


Adaptive Correction (TSS) defined


## Step 11 — Evaluation (Accuracy + AUROC + Forgetting)

**Metrics** (from ARC paper):
- **Average Accuracy (AB)** = mean accuracy across all tasks after final training
- **Forgetting (F)** = mean drop from peak accuracy on each past task
- **AUROC** = per-class one-vs-rest AUC, averaged over seen classes
- **BWT** (Backward Transfer) = mean change in past-task accuracy after new task learning
- **FWT** (Forward Transfer) = how much prior tasks help new task performance

**ARC evaluation design**:
- ARC-adapted classifier (`clf_arc`) persists across all task evaluation calls within a test stream
- Retention weight updates accumulate — not reset per evaluation
- AUROC computed from ARC-adapted logits (not baseline logits)


In [18]:
def compute_auroc(y_t, probs_np, seen_cls, n_total_cls):
    """Per-class OVR AUROC averaged over valid seen classes."""
    valid_seen = [c for c in seen_cls if c < n_total_cls]
    if len(valid_seen) < 2:
        return float('nan')
    y_t_arr    = np.asarray(y_t)
    y_bin      = np.stack([(y_t_arr == c).astype(int) for c in valid_seen], axis=1)
    probs_seen = probs_np[:, valid_seen]
    probs_seen = probs_seen / (probs_seen.sum(axis=1, keepdims=True) + 1e-8)
    auroc_list = []
    for col_i in range(len(valid_seen)):
        col_true = y_bin[:, col_i]
        col_prob = probs_seen[:, col_i]
        if col_true.sum() == 0 or col_true.sum() == len(col_true):
            continue
        try:
            auroc_list.append(roc_auc_score(col_true, col_prob))
        except Exception:
            pass
    return float(np.mean(auroc_list)) if auroc_list else float('nan')


def evaluate_task(
    clf, X_test, y_test, seen_cls,
    task_id=None, n_cpt=None, epsilon=None, theta=None, temp=None,
    arc_lr=None, use_arc=False, device='cpu', persistent_clf_arc=None,
):
    """
    Evaluate classifier on test samples from seen_cls.

    FIX-3 (v8): persistent_clf_arc passed in — retention updates accumulate
    across all evaluation calls. Old code deepcopied clf inside here → all
    updates lost between evaluations.

    FIX-2 (v8): batch_theta computed once over full test batch before loop.
    FIX-1 (v8): OTD uses corrected Assumption 1/2 logic (see out_of_task_detection).
    """
    mask = np.isin(y_test, seen_cls)
    X_t, y_t = X_test[mask], y_test[mask]
    if len(X_t) == 0:
        return {'accuracy': 0.0, 'auroc': float('nan')}

    Xt = torch.tensor(X_t, dtype=torch.float32).to(device)
    clf.eval()
    with torch.no_grad():
        logits_all = clf(Xt)
    n_total_cls = logits_all.shape[1]

    if use_arc and task_id is not None and task_id > 0:
        clf_arc = persistent_clf_arc if persistent_clf_arc is not None else copy.deepcopy(clf)

        # FIX-2: compute adaptive theta ONCE from full batch
        with torch.no_grad():
            clf_arc.eval()
            logits_for_theta = clf_arc(Xt)
        batch_theta = compute_batch_theta(logits_for_theta, task_id, n_cpt, theta)

        otd_counts = {'retention': 0, 'correction': 0, 'current': 0}
        preds, adapted_logits_list = [], []

        for i in range(len(Xt)):
            xi = Xt[i:i+1]
            with torch.no_grad():
                clf_arc.eval()
                li_otd = clf_arc(xi)

            decisions, _, pred_i = out_of_task_detection(
                li_otd, task_id, n_cpt, epsilon, theta, batch_theta=batch_theta
            )
            decision     = decisions[0]
            pseudo_label = pred_i[0].item()
            otd_counts[decision] += 1

            if decision == 'retention':
                adaptive_retention_step(clf_arc, xi, pseudo_label, arc_lr, device)
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                pred_final = li_final.argmax(1).item()

            elif decision == 'correction':
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                _, pred_final = compute_tss(li_final[0], task_id, n_cpt, temp)

            else:
                li_final   = li_otd
                pred_final = pseudo_label

            preds.append(pred_final)
            adapted_logits_list.append(li_final.detach())

        preds = np.array(preds)
        adapted_logits = torch.cat(adapted_logits_list, dim=0)
        with torch.no_grad():
            probs_np = F.softmax(adapted_logits, dim=-1).cpu().numpy()

        n = len(Xt)
        print(f'    OTD → retention:{otd_counts["retention"]}/{n}  '
              f'correction:{otd_counts["correction"]}/{n}  '
              f'current:{otd_counts["current"]}/{n}  (batch_θ={batch_theta:.3f})')

    else:
        with torch.no_grad():
            preds    = logits_all.argmax(1).cpu().numpy()
            probs_np = F.softmax(logits_all, dim=-1).cpu().numpy()

    accuracy = float((preds == y_t).mean())
    auroc    = compute_auroc(y_t, probs_np, seen_cls, n_total_cls)
    return {'accuracy': accuracy, 'auroc': auroc, 'preds': preds, 'labels': y_t}


print('evaluate_task defined')


evaluate_task defined


## Step 12 — Full CIL Pipeline

In [45]:
def evaluate_task(
    clf, X_test, y_test, seen_cls,
    current_task_id=None,   # current trained task (OTD boundary ke liye)
    eval_task_id=None,      # task being evaluated (guard ke liye)
    n_cpt=None, epsilon=None, theta=None, temp=None,
    arc_lr=None, use_arc=False, device='cpu', persistent_clf_arc=None,
):
    mask = np.isin(y_test, seen_cls)
    X_t, y_t = X_test[mask], y_test[mask]
    if len(X_t) == 0:
        return {'accuracy': 0.0, 'auroc': float('nan')}

    Xt = torch.tensor(X_t, dtype=torch.float32).to(device)
    clf.eval()
    with torch.no_grad():
        logits_all = clf(Xt)
    n_total_cls = logits_all.shape[1]

    # ARC: only when there are past tasks (current_task_id > 0)
    if use_arc and current_task_id is not None and current_task_id > 0:
        # Fresh snapshot per evaluation — prevents cross-eval contamination
        # persistent_clf_arc = task boundary pe reset hota hai (task loop mein)
        # per-evaluation deepcopy se Task 0 ke retention updates Task 1 ko corrupt nahi karte
        clf_arc = copy.deepcopy(persistent_clf_arc) if persistent_clf_arc is not None \
                  else copy.deepcopy(clf)

        with torch.no_grad():
            clf_arc.eval()
            logits_for_theta = clf_arc(Xt)
        batch_theta = compute_batch_theta(
            logits_for_theta, current_task_id, n_cpt, theta)

        otd_counts = {'retention': 0, 'correction': 0, 'current': 0}
        preds, adapted_logits_list = [], []

        for i in range(len(Xt)):
            xi = Xt[i:i+1]
            with torch.no_grad():
                clf_arc.eval()
                li_otd = clf_arc(xi)

            decisions, _, pred_i = out_of_task_detection(
                li_otd, current_task_id, n_cpt, epsilon, theta,
                batch_theta=batch_theta)
            decision     = decisions[0]
            pseudo_label = pred_i[0].item()
            otd_counts[decision] += 1

            if decision == 'retention':
                adaptive_retention_step(clf_arc, xi, pseudo_label, arc_lr, device)
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                pred_final = li_final.argmax(1).item()
            elif decision == 'correction':
                with torch.no_grad():
                    clf_arc.eval()
                    li_final = clf_arc(xi)
                _, pred_final = compute_tss(
                    li_final[0], current_task_id, n_cpt, temp)
            else:
                li_final   = li_otd
                pred_final = pseudo_label

            preds.append(pred_final)
            adapted_logits_list.append(li_final.detach())

        preds = np.array(preds)
        adapted_logits = torch.cat(adapted_logits_list, dim=0)
        with torch.no_grad():
            probs_np = F.softmax(adapted_logits, dim=-1).cpu().numpy()

        n = len(Xt)
        print(f'    OTD → retention:{otd_counts["retention"]}/{n}  '
              f'correction:{otd_counts["correction"]}/{n}  '
              f'current:{otd_counts["current"]}/{n}  '
              f'(batch_θ={batch_theta:.3f})')
    else:
        with torch.no_grad():
            preds    = logits_all.argmax(1).cpu().numpy()
            probs_np = F.softmax(logits_all, dim=-1).cpu().numpy()

    accuracy = float((preds == y_t).mean())
    auroc    = compute_auroc(y_t, probs_np, seen_cls, n_total_cls)
    return {'accuracy': accuracy, 'auroc': auroc, 'preds': preds, 'labels': y_t}


print('evaluate_task defined')


def run_cil_pipeline(
    tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
    feat_dim, label_map, cfg, device, use_arc=False, name='Dataset'
):
    print(f'\n{"="*60}')
    print(f'  {name} CIL Pipeline  |  ARC={use_arc}')
    print(f'{"="*60}')
    print('  [Replay OFF] — memory-free for both baseline and ARC')

    clf  = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    R    = defaultdict(dict)
    peak_acc = {}
    rows     = []
    persistent_clf_arc = None

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        print(f'\n-- Task {tid} | New classes: {[task["class_names"][c] for c in task["new_classes"]]} --')

        if tid > 0:
            clf.grow(len(task['new_classes']), device)

        clf = train_classifier_on_task(
            clf, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device,
            es_patience  = cfg.get('es_patience', 10),
            es_min_delta = cfg.get('es_min_delta', 1e-4),
        )

        if use_arc:
            persistent_clf_arc = copy.deepcopy(clf)

        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']

            res = evaluate_task(
                clf, X_te, y_te, prev_seen,
                current_task_id    = tid,
                eval_task_id       = prev_tid,
                n_cpt              = cfg['classes_per_task'],
                epsilon            = cfg['arc_epsilon'],
                theta              = cfg['arc_theta'],
                temp               = cfg['arc_temp'],
                arc_lr             = cfg['arc_lr'],
                use_arc            = use_arc,
                device             = device,
                persistent_clf_arc = persistent_clf_arc,
            )
            R[tid][prev_tid] = res

            if prev_tid == tid:
                peak_acc[tid] = res['accuracy']

            auroc_str = f"{res['auroc']:.4f}" if not np.isnan(res['auroc']) else 'nan'
            print(f'  Eval Task {prev_tid} → acc={res["accuracy"]:.4f}  auroc={auroc_str}')

    n_tasks   = len(tasks)
    final_tid = n_tasks - 1
    acc_list, auroc_list, forget_list = [], [], []

    for prev_tid in range(n_tasks):
        final_acc   = R[final_tid][prev_tid]['accuracy']
        final_auroc = R[final_tid][prev_tid]['auroc']
        acc_list.append(final_acc)
        auroc_list.append(final_auroc)
        if prev_tid < final_tid:
            forget_list.append(peak_acc[prev_tid] - final_acc)

        rows.append({
            'task_id'    : prev_tid,
            'class_names': str(list(tasks[prev_tid]['class_names'].values())),
            'final_acc'  : round(final_acc, 4),
            'final_auroc': round(final_auroc, 4) if not np.isnan(final_auroc) else float('nan'),
            'peak_acc'   : round(peak_acc.get(prev_tid, final_acc), 4),
            'forgetting' : round(peak_acc.get(prev_tid, final_acc) - final_acc, 4),
        })

    avg_acc    = float(np.mean(acc_list))
    forgetting = float(np.mean(forget_list)) if forget_list else 0.0
    avg_auroc  = float(np.nanmean(auroc_list))

    print(f'\n{"─"*50}')
    print(f'  Average Accuracy (AB) : {avg_acc:.4f}')
    print(f'  Forgetting (F)        : {forgetting:.4f}')
    print(f'  Average AUROC         : {avg_auroc:.4f}')
    print(f'{"─"*50}')

    return pd.DataFrame(rows), avg_acc, forgetting, avg_auroc

print('CIL pipeline defined')

evaluate_task defined
CIL pipeline defined


## Step 13 — Run: ChEMBL (Baseline)

In [46]:
chembl_res_no_arc, chembl_AB_no_arc, chembl_F_no_arc, chembl_AUROC_no_arc = run_cil_pipeline(
    tasks=chembl_tasks,
    X_tr=chembl_X_tr, y_tr=chembl_y_tr,
    X_val=chembl_X_vl, y_val=chembl_y_vl,
    X_te=chembl_X_te, y_te=chembl_y_te,
    feat_dim=FEAT_DIM, label_map=chembl_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='ChEMBL'
)
print('\nChEMBL Results (No ARC):')
print(chembl_res_no_arc.to_string(index=False))



  ChEMBL CIL Pipeline  |  ARC=False
  [Replay OFF] — memory-free for both baseline and ARC

-- Task 0 | New classes: ['Target-0', 'Target-1', 'Target-2'] --
  Eval Task 0 → acc=0.3742  auroc=0.5551

-- Task 1 | New classes: ['Target-3', 'Target-4', 'Target-5'] --
  Eval Task 0 → acc=0.2371  auroc=0.5551
  Eval Task 1 → acc=0.2211  auroc=0.5621

-- Task 2 | New classes: ['Target-6', 'Target-7', 'Target-8'] --
  Eval Task 0 → acc=0.1798  auroc=0.5610
  Eval Task 1 → acc=0.1678  auroc=0.5644
  Eval Task 2 → acc=0.1695  auroc=0.5773

-- Task 3 | New classes: ['Target-9', 'Target-10', 'Target-11'] --
  Eval Task 0 → acc=0.2090  auroc=0.5613
  Eval Task 1 → acc=0.1538  auroc=0.5668
  Eval Task 2 → acc=0.1351  auroc=0.5793
  Eval Task 3 → acc=0.1325  auroc=0.6018

-- Task 4 | New classes: ['Target-12', 'Target-13', 'Target-14'] --
  Eval Task 0 → acc=0.1843  auroc=0.5542
  Eval Task 1 → acc=0.1369  auroc=0.5627
  Eval Task 2 → acc=0.1092  auroc=0.5737
  Eval Task 3 → acc=0.1002  auroc=0.5994

## Step 14 — Run: ChEMBL + ARC

In [47]:
chembl_res_arc, chembl_AB_arc, chembl_F_arc, chembl_AUROC_arc = run_cil_pipeline(
    tasks=chembl_tasks,
    X_tr=chembl_X_tr, y_tr=chembl_y_tr,
    X_val=chembl_X_vl, y_val=chembl_y_vl,
    X_te=chembl_X_te, y_te=chembl_y_te,
    feat_dim=FEAT_DIM, label_map=chembl_label_map,
    cfg=CFG, device=DEVICE, use_arc=True, name='ChEMBL + ARC'
)
print('\nChEMBL Results (With ARC):')
print(chembl_res_arc.to_string(index=False))



  ChEMBL + ARC CIL Pipeline  |  ARC=True
  [Replay OFF] — memory-free for both baseline and ARC

-- Task 0 | New classes: ['Target-0', 'Target-1', 'Target-2'] --
  Eval Task 0 → acc=0.3865  auroc=0.5511

-- Task 1 | New classes: ['Target-3', 'Target-4', 'Target-5'] --
    OTD → retention:885/890  correction:0/890  current:5/890  (batch_θ=0.829)
  Eval Task 0 → acc=0.3247  auroc=0.2180
    OTD → retention:1777/1782  correction:0/1782  current:5/1782  (batch_θ=0.856)
  Eval Task 1 → acc=0.1622  auroc=0.3532

-- Task 2 | New classes: ['Target-6', 'Target-7', 'Target-8'] --
    OTD → retention:878/890  correction:0/890  current:12/890  (batch_θ=0.800)
  Eval Task 0 → acc=0.3180  auroc=0.2201
    OTD → retention:1770/1782  correction:0/1782  current:12/1782  (batch_θ=0.800)
  Eval Task 1 → acc=0.1588  auroc=0.3635
    OTD → retention:2661/2673  correction:0/2673  current:12/2673  (batch_θ=0.800)
  Eval Task 2 → acc=0.1059  auroc=0.4201

-- Task 3 | New classes: ['Target-9', 'Target-10', 'T

## Step 15 — Run: BindingDB (Baseline)

In [29]:
bindingdb_res_no_arc, bdb_AB_no_arc, bdb_F_no_arc, bdb_AUROC_no_arc = run_cil_pipeline(
    tasks=bindingdb_tasks,
    X_tr=bindingdb_X_tr, y_tr=bindingdb_y_tr,
    X_val=bindingdb_X_vl, y_val=bindingdb_y_vl,
    X_te=bindingdb_X_te, y_te=bindingdb_y_te,
    feat_dim=FEAT_DIM, label_map=bindingdb_label_map,
    cfg=CFG, device=DEVICE, use_arc=False, name='BindingDB'
)
print('\nBindingDB Results (No ARC):')
print(bindingdb_res_no_arc.to_string(index=False))



  BindingDB CIL Pipeline  |  ARC=False
  [Replay ON]  — Baseline uses experience replay

-- Task 0 | New classes: ['Target-0', 'Target-1', 'Target-2'] --
  Eval Task 0 → acc=0.3681  auroc=0.5621

-- Task 1 | New classes: ['Target-3', 'Target-4', 'Target-5'] --
  Eval Task 0 → acc=0.2265  auroc=0.5132
  Eval Task 1 → acc=0.1905  auroc=0.5430

-- Task 2 | New classes: ['Target-6', 'Target-7', 'Target-8'] --
  Eval Task 0 → acc=0.1880  auroc=0.5432
  Eval Task 1 → acc=0.1849  auroc=0.5410
  Eval Task 2 → acc=0.1437  auroc=0.5545

-- Task 3 | New classes: ['Target-9', 'Target-10', 'Target-11'] --
  Eval Task 0 → acc=0.1823  auroc=0.5582
  Eval Task 1 → acc=0.1385  auroc=0.5400
  Eval Task 2 → acc=0.1290  auroc=0.5452
  Eval Task 3 → acc=0.1192  auroc=0.5673

-- Task 4 | New classes: ['Target-12', 'Target-13', 'Target-14'] --
  Eval Task 0 → acc=0.1518  auroc=0.5517
  Eval Task 1 → acc=0.1221  auroc=0.5450
  Eval Task 2 → acc=0.1060  auroc=0.5510
  Eval Task 3 → acc=0.1000  auroc=0.5657
  

## Step 16 — Run: BindingDB + ARC

In [30]:
bindingdb_res_arc, bdb_AB_arc, bdb_F_arc, bdb_AUROC_arc = run_cil_pipeline(
    tasks=bindingdb_tasks,
    X_tr=bindingdb_X_tr, y_tr=bindingdb_y_tr,
    X_val=bindingdb_X_vl, y_val=bindingdb_y_vl,
    X_te=bindingdb_X_te, y_te=bindingdb_y_te,
    feat_dim=FEAT_DIM, label_map=bindingdb_label_map,
    cfg=CFG, device=DEVICE, use_arc=True, name='BindingDB + ARC'
)
print('\nBindingDB Results (With ARC):')
print(bindingdb_res_arc.to_string(index=False))



  BindingDB + ARC CIL Pipeline  |  ARC=True
  [Replay OFF] — ARC run is memory-free (paper Sec 3, memory-free design)

-- Task 0 | New classes: ['Target-0', 'Target-1', 'Target-2'] --
  Eval Task 0 → acc=0.3862  auroc=0.5643

-- Task 1 | New classes: ['Target-3', 'Target-4', 'Target-5'] --
  Eval Task 0 → acc=0.1721  auroc=0.5646
    OTD → retention:1760/1769  correction:0/1769  current:9/1769  (batch_θ=0.983)
  Eval Task 1 → acc=0.1594  auroc=0.3621

-- Task 2 | New classes: ['Target-6', 'Target-7', 'Target-8'] --
  Eval Task 0 → acc=0.1302  auroc=0.5593
    OTD → retention:1755/1769  correction:1/1769  current:13/1769  (batch_θ=1.109)
  Eval Task 1 → acc=0.1577  auroc=0.3614
    OTD → retention:2652/2652  correction:0/2652  current:0/2652  (batch_θ=0.800)
  Eval Task 2 → acc=0.1101  auroc=0.5121

-- Task 3 | New classes: ['Target-9', 'Target-10', 'Target-11'] --
  Eval Task 0 → acc=0.1540  auroc=0.5590
    OTD → retention:1753/1769  correction:1/1769  current:15/1769  (batch_θ=1.083

## Step 17 — Final Comparison Table & BWT/FWT

In [24]:
def compute_bwt_fwt(tasks, X_tr, y_tr, X_val, y_val, X_te, y_te,
                    feat_dim, cfg, device, use_arc=False):
    """
    BWT (Backward Transfer): R(T,i) - R(i,i) averaged over past tasks.
      Negative = forgetting; Positive = backward beneficial transfer.
    FWT (Forward Transfer): R(i,i) - chance for each task i.
      Positive = forward transfer (prior tasks help).
    """
    n_tasks = len(tasks)
    if n_tasks < 2: return float('nan'), float('nan')

    R = {}
    clf_tmp = ExpandingLinearHead(feat_dim, tasks[0]['n_classes']).to(device)
    peak_acc_tmp = {}

    for task in tasks:
        tid      = task['task_id']
        seen_cls = task['all_classes']
        if tid > 0:
            clf_tmp.grow(len(task['new_classes']), device)
        clf_tmp = train_classifier_on_task(
            clf_tmp, X_tr, y_tr, X_val, y_val,
            seen_cls, cfg['clf_epochs'], cfg['clf_lr'], cfg['clf_batch'], device
        )
        R[tid] = {}
        for prev_task in tasks[:tid + 1]:
            prev_tid  = prev_task['task_id']
            prev_seen = prev_task['all_classes']
            res = evaluate_task(
                clf_tmp, X_te, y_te, prev_seen,
                task_id=tid, n_cpt=cfg['classes_per_task'],
                epsilon=cfg['arc_epsilon'], theta=cfg['arc_theta'],
                temp=cfg['arc_temp'], arc_lr=cfg['arc_lr'],
                use_arc=use_arc, device=device
            )
            R[tid][prev_tid] = res['accuracy']
            if prev_tid == tid:
                peak_acc_tmp[tid] = res['accuracy']

    bwt_vals = [R[n_tasks - 1][i] - peak_acc_tmp[i] for i in range(n_tasks - 1)]
    bwt = float(np.mean(bwt_vals))

    fwt_vals = []
    for i in range(1, n_tasks):
        n_cls_i  = tasks[i]['n_classes']
        chance_i = 1.0 / n_cls_i
        fwt_vals.append(R[i][i] - chance_i)
    fwt = float(np.mean(fwt_vals)) if fwt_vals else float('nan')
    return bwt, fwt


# ── BWT/FWT for ChEMBL ────────────────────────────────────────────────────────
print('Computing BWT/FWT for ChEMBL...')
chembl_BWT_no_arc, chembl_FWT_no_arc = compute_bwt_fwt(
    chembl_tasks, chembl_X_tr, chembl_y_tr, chembl_X_vl, chembl_y_vl,
    chembl_X_te, chembl_y_te, FEAT_DIM, CFG, DEVICE, use_arc=False)
print(f'  ChEMBL No ARC → BWT={chembl_BWT_no_arc:.4f}  FWT={chembl_FWT_no_arc:.4f}')

chembl_BWT_arc, chembl_FWT_arc = compute_bwt_fwt(
    chembl_tasks, chembl_X_tr, chembl_y_tr, chembl_X_vl, chembl_y_vl,
    chembl_X_te, chembl_y_te, FEAT_DIM, CFG, DEVICE, use_arc=True)
print(f'  ChEMBL + ARC  → BWT={chembl_BWT_arc:.4f}  FWT={chembl_FWT_arc:.4f}')

# ── BWT/FWT for BindingDB ─────────────────────────────────────────────────────
print('\nComputing BWT/FWT for BindingDB...')
bdb_BWT_no_arc, bdb_FWT_no_arc = compute_bwt_fwt(
    bindingdb_tasks, bindingdb_X_tr, bindingdb_y_tr, bindingdb_X_vl, bindingdb_y_vl,
    bindingdb_X_te, bindingdb_y_te, FEAT_DIM, CFG, DEVICE, use_arc=False)
print(f'  BindingDB No ARC → BWT={bdb_BWT_no_arc:.4f}  FWT={bdb_FWT_no_arc:.4f}')

bdb_BWT_arc, bdb_FWT_arc = compute_bwt_fwt(
    bindingdb_tasks, bindingdb_X_tr, bindingdb_y_tr, bindingdb_X_vl, bindingdb_y_vl,
    bindingdb_X_te, bindingdb_y_te, FEAT_DIM, CFG, DEVICE, use_arc=True)
print(f'  BindingDB + ARC  → BWT={bdb_BWT_arc:.4f}  FWT={bdb_FWT_arc:.4f}')


Computing BWT/FWT for ChEMBL...
  ChEMBL No ARC → BWT=-0.0843  FWT=0.0583
    OTD → retention:611/890  correction:0/890  current:279/890  (batch_θ=0.400)
    OTD → retention:1503/1782  correction:0/1782  current:279/1782  (batch_θ=0.400)
    OTD → retention:660/890  correction:0/890  current:230/890  (batch_θ=0.400)
    OTD → retention:1552/1782  correction:0/1782  current:230/1782  (batch_θ=0.400)
    OTD → retention:2443/2673  correction:0/2673  current:230/2673  (batch_θ=0.400)
    OTD → retention:548/890  correction:0/890  current:342/890  (batch_θ=0.400)
    OTD → retention:1440/1782  correction:0/1782  current:342/1782  (batch_θ=0.400)
    OTD → retention:2331/2673  correction:0/2673  current:342/2673  (batch_θ=0.400)
    OTD → retention:3221/3563  correction:0/3563  current:342/3563  (batch_θ=0.400)
    OTD → retention:704/890  correction:0/890  current:186/890  (batch_θ=0.400)
    OTD → retention:1596/1782  correction:0/1782  current:186/1782  (batch_θ=0.400)
    OTD → retentio

In [ ]:
# ── Final Comparison Table ────────────────────────────────────────────────────
comparison = pd.DataFrame([
    {'Dataset': 'ChEMBL',   'Method': 'Baseline (replay)',
     'AB': round(chembl_AB_no_arc, 4), 'F': round(chembl_F_no_arc, 4),
     'AUROC': round(chembl_AUROC_no_arc, 4),
     'BWT': round(chembl_BWT_no_arc, 4), 'FWT': round(chembl_FWT_no_arc, 4)},
    {'Dataset': 'ChEMBL',   'Method': '+ ARC (memory-free)',
     'AB': round(chembl_AB_arc, 4), 'F': round(chembl_F_arc, 4),
     'AUROC': round(chembl_AUROC_arc, 4),
     'BWT': round(chembl_BWT_arc, 4), 'FWT': round(chembl_FWT_arc, 4)},
    {'Dataset': 'BindingDB', 'Method': 'Baseline (replay)',
     'AB': round(bdb_AB_no_arc, 4), 'F': round(bdb_F_no_arc, 4),
     'AUROC': round(bdb_AUROC_no_arc, 4),
     'BWT': round(bdb_BWT_no_arc, 4), 'FWT': round(bdb_FWT_no_arc, 4)},
    {'Dataset': 'BindingDB', 'Method': '+ ARC (memory-free)',
     'AB': round(bdb_AB_arc, 4), 'F': round(bdb_F_arc, 4),
     'AUROC': round(bdb_AUROC_arc, 4),
     'BWT': round(bdb_BWT_arc, 4), 'FWT': round(bdb_FWT_arc, 4)},
])

print('=' * 80)
print('                     FINAL COMPARISON TABLE')
print('=' * 80)
print(comparison.to_string(index=False))
print()
print('ARC Improvement Delta:')
print(f'  ChEMBL   → ΔAB={chembl_AB_arc-chembl_AB_no_arc:+.4f}  ΔF={chembl_F_arc-chembl_F_no_arc:+.4f}  ΔAUROC={chembl_AUROC_arc-chembl_AUROC_no_arc:+.4f}  ΔBWT={chembl_BWT_arc-chembl_BWT_no_arc:+.4f}')
print(f'  BindingDB → ΔAB={bdb_AB_arc-bdb_AB_no_arc:+.4f}  ΔF={bdb_F_arc-bdb_F_no_arc:+.4f}  ΔAUROC={bdb_AUROC_arc-bdb_AUROC_no_arc:+.4f}  ΔBWT={bdb_BWT_arc-bdb_BWT_no_arc:+.4f}')
print('=' * 80)
print('  BWT: negative=forgetting  positive=backward beneficial transfer')
print('  FWT: positive=forward transfer  (prior tasks help new ones)')
comparison


## Step 18 — Visualisation

In [ ]:
def plot_per_task_metrics(res_no_arc, res_arc, dataset_name):
    """Bar chart comparing per-task accuracy and forgetting: baseline vs ARC."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{dataset_name}: Per-Task Metrics — Baseline vs ARC', fontsize=13)
    x = np.arange(len(res_no_arc))
    w = 0.35

    # Accuracy
    axes[0].bar(x - w/2, res_no_arc['final_acc'], w, label='Baseline', color='steelblue', alpha=0.85)
    axes[0].bar(x + w/2, res_arc['final_acc'],    w, label='+ ARC',    color='coral',     alpha=0.85)
    axes[0].set_xlabel('Task'); axes[0].set_ylabel('Final Accuracy')
    axes[0].set_title('Final Accuracy per Task'); axes[0].legend()
    axes[0].set_xticks(x); axes[0].set_xticklabels([f'T{i}' for i in range(len(x))])
    axes[0].set_ylim(0, 1)

    # Forgetting
    axes[1].bar(x - w/2, res_no_arc['forgetting'], w, label='Baseline', color='steelblue', alpha=0.85)
    axes[1].bar(x + w/2, res_arc['forgetting'],    w, label='+ ARC',    color='coral',     alpha=0.85)
    axes[1].set_xlabel('Task'); axes[1].set_ylabel('Forgetting')
    axes[1].set_title('Forgetting per Task'); axes[1].legend()
    axes[1].set_xticks(x); axes[1].set_xticklabels([f'T{i}' for i in range(len(x))])

    plt.tight_layout()
    plt.savefig(os.path.join(CFG['output_dir'], f'{dataset_name.lower()}_per_task.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {dataset_name.lower()}_per_task.png')


def plot_summary_comparison(comparison_df):
    """Grouped bar chart of AB, F, AUROC across datasets and methods."""
    metrics  = ['AB', 'F', 'AUROC']
    datasets = ['ChEMBL', 'BindingDB']
    methods  = ['Baseline (replay)', '+ ARC (memory-free)']
    colors   = ['steelblue', 'coral']

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('ARC vs Baseline: ChEMBL & BindingDB', fontsize=13)

    for ax, metric in zip(axes, metrics):
        x = np.arange(len(datasets))
        for j, (method, color) in enumerate(zip(methods, colors)):
            vals = [comparison_df[(comparison_df['Dataset'] == ds) &
                                  (comparison_df['Method'] == method)][metric].values[0]
                    for ds in datasets]
            ax.bar(x + j * 0.35, vals, 0.35, label=method, color=color, alpha=0.85)
        ax.set_title(f'{metric}'); ax.legend(fontsize=8)
        ax.set_xticks(x + 0.175); ax.set_xticklabels(datasets)
        if metric in ('AB', 'AUROC'): ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(os.path.join(CFG['output_dir'], 'summary_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: summary_comparison.png')


plot_per_task_metrics(chembl_res_no_arc, chembl_res_arc, 'ChEMBL')
plot_per_task_metrics(bindingdb_res_no_arc, bindingdb_res_arc, 'BindingDB')
plot_summary_comparison(comparison)


## Step 19 — Save Results

In [ ]:
OUT = CFG['output_dir']
chembl_res_no_arc.to_csv  (os.path.join(OUT, 'chembl_no_arc.csv'),   index=False)
chembl_res_arc.to_csv     (os.path.join(OUT, 'chembl_arc.csv'),      index=False)
bindingdb_res_no_arc.to_csv(os.path.join(OUT, 'bindingdb_no_arc.csv'), index=False)
bindingdb_res_arc.to_csv  (os.path.join(OUT, 'bindingdb_arc.csv'),   index=False)
comparison.to_csv         (os.path.join(OUT, 'comparison.csv'),      index=False)

meta = {
    'chembl'   : {'no_arc': {'AB': chembl_AB_no_arc, 'F': chembl_F_no_arc, 'AUROC': chembl_AUROC_no_arc,
                              'BWT': chembl_BWT_no_arc, 'FWT': chembl_FWT_no_arc},
                  'arc'   : {'AB': chembl_AB_arc,    'F': chembl_F_arc,    'AUROC': chembl_AUROC_arc,
                              'BWT': chembl_BWT_arc,    'FWT': chembl_FWT_arc}},
    'bindingdb': {'no_arc': {'AB': bdb_AB_no_arc, 'F': bdb_F_no_arc, 'AUROC': bdb_AUROC_no_arc,
                              'BWT': bdb_BWT_no_arc, 'FWT': bdb_FWT_no_arc},
                  'arc'   : {'AB': bdb_AB_arc,    'F': bdb_F_arc,    'AUROC': bdb_AUROC_arc,
                              'BWT': bdb_BWT_arc,    'FWT': bdb_FWT_arc}},
    'config'   : {k: str(v) for k, v in CFG.items()},
}
with open(os.path.join(OUT, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f'All results saved to: {OUT}/')
print('  chembl_no_arc.csv  |  chembl_arc.csv')
print('  bindingdb_no_arc.csv  |  bindingdb_arc.csv')
print('  comparison.csv  |  metadata.json')
